# Inverse Heat Conduction via Enzyme AD

In this tutorial, you will learn how to:

1. **Build a Tesseract** that wraps a Fortran heat equation solver differentiated by [Enzyme](https://enzyme.mit.edu/) (an LLVM-based automatic differentiation compiler plugin)
2. **Embed it in a JAX pipeline** via [tesseract-jax](https://github.com/pasteurlabs/tesseract-jax), making it a native JAX-differentiable function
3. **Solve an inverse problem**: recover the thermal diffusivity of a material from temperature observations, using `jax.grad` to differentiate *through* the compiled Fortran solver

## Why this matters

Scientific computing is full of Fortran and C code that researchers need gradients of -- for optimization, inverse problems, uncertainty quantification, or integration with ML pipelines. The traditional options are painful:

- **Hand-written adjoints**: months of expert effort, error-prone, a maintenance nightmare
- **Finite differences**: slow ($O(n)$ evaluations per gradient), inaccurate (truncation vs. roundoff tradeoff)
- **Rewrite in JAX/PyTorch**: impractical for existing codebases

Enzyme offers a fourth option: **automatic differentiation at the LLVM IR level**. It takes compiled code and synthesizes exact derivative functions -- no source modifications, no manual adjoints, no approximation. And because it operates on LLVM IR, it works with any language that compiles to it: Fortran, C, C++, Rust, and more.

In this demo, we differentiate a Fortran heat equation solver using Enzyme, package it as a [Tesseract](https://github.com/pasteurlabs/tesseract), and use the exact gradients to solve an inverse problem entirely within JAX.

In [ ]:
# Install additional requirements for this notebook
%pip install tesseract-jax -q

## Step 1: Build and serve the Enzyme AD Tesseract

The `enzyme-ad` Tesseract wraps a Fortran implementation of a single explicit Euler step of the 1D heat equation:

$$\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial x^2}$$

The Fortran source (`heat_step.f90`) is just ~30 lines -- a simple finite difference stencil:

```fortran
do i = 2, n - 1
    T_out(i) = T_in(i) + r * (T_in(i-1) - 2.0d0*T_in(i) + T_in(i+1))
end do
```

During `tesseract build`, the compilation pipeline runs:

```
Fortran --> LFortran --> LLVM IR --> Enzyme AD pass --> libheat_ad.so
```

Enzyme analyzes the LLVM IR and generates both forward-mode (JVP) and reverse-mode (VJP) derivative functions automatically. The resulting shared library has three entry points: `heat_step_forward`, `heat_step_jvp`, and `heat_step_vjp`.

In [ ]:
%%bash
tesseract build ../../examples/enzyme_ad/

In [ ]:
from tesseract_core import Tesseract

heat_tesseract = Tesseract.from_image("enzyme-ad")
heat_tesseract.serve()

## Step 2: Test a forward evaluation

Before optimizing, let's verify the Tesseract works. We set up a temperature profile (a Gaussian bump on a uniform grid) and run one heat equation step.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from tesseract_jax import apply_tesseract

jax.config.update("jax_enable_x64", True)

# Grid setup
n_points = 50
dx = 1.0 / (n_points - 1)
x = jnp.linspace(0.0, 1.0, n_points)

# Initial temperature: Gaussian bump
T_init = jnp.exp(-((x - 0.5) ** 2) / (2 * 0.05**2))
# Fix boundary conditions to zero
T_init = T_init.at[0].set(0.0).at[-1].set(0.0)

# Physical parameters
alpha_true = 0.02  # true thermal diffusivity
dt = 0.0001


def heat_step(T_in, alpha):
    """Run one heat equation step through the Enzyme-differentiated Fortran solver."""
    result = apply_tesseract(
        heat_tesseract,
        {"T_in": T_in, "alpha": alpha, "dx": dx, "dt": dt},
    )
    return result["T_out"]


# Run one step
T_after_one = heat_step(T_init, alpha_true)

plt.figure(figsize=(8, 4))
plt.plot(x, T_init, label="Initial", linewidth=2)
plt.plot(x, T_after_one, label="After 1 step", linewidth=2)
plt.xlabel("x")
plt.ylabel("T")
plt.legend()
plt.title("Single heat equation step (Enzyme-differentiated Fortran solver)")
plt.tight_layout()

## Step 3: Run multiple steps to generate synthetic observations

For the inverse problem, we need "observed" data. We'll run the solver forward for many steps with the true $\alpha$ to produce a final temperature profile, then pretend we only know this final profile and try to recover $\alpha$.

In [ ]:
def simulate(T_init, alpha, n_steps):
    """Run the heat equation forward for n_steps."""
    T = T_init
    for _ in range(n_steps):
        T = heat_step(T, alpha)
    return T


# Generate "observed" data with the true alpha
n_steps = 200
T_observed = jax.jit(simulate, static_argnums=(2,))(T_init, alpha_true, n_steps)

plt.figure(figsize=(8, 4))
plt.plot(x, T_init, label="Initial condition", linewidth=2)
plt.plot(x, T_observed, "k--", label=f"Observed (after {n_steps} steps)", linewidth=2)
plt.xlabel("x")
plt.ylabel("T")
plt.legend()
plt.title("Forward simulation with true $\\alpha$")
plt.tight_layout()

## Step 4: Solve the inverse problem

Now the fun part. We want to recover $\alpha$ from the observed final temperature profile. We define the loss as the mean squared error between the simulated and observed profiles:

$$\mathcal{L}(\alpha) = \text{MSE}\big(T_{\text{sim}}(\alpha),\; T_{\text{obs}}\big)$$

Because the Tesseract exposes Enzyme-generated VJPs, `jax.grad` can differentiate through the entire multi-step simulation -- computing $\partial \mathcal{L}/\partial \alpha$ by backpropagating through hundreds of Fortran solver calls. We use L-BFGS-B from `scipy.optimize` to find the optimal $\alpha$.

In [ ]:
def loss_fn(alpha):
    """MSE between simulated and observed temperature profiles."""
    T_sim = simulate(T_init, alpha, n_steps)
    return jnp.mean((T_sim - T_observed) ** 2)


# Verify gradients work
grad_fn = jax.jit(jax.value_and_grad(loss_fn))
test_loss, test_grad = grad_fn(jnp.float64(0.05))
print(f"Loss at alpha=0.05: {test_loss:.6e}")
print(f"Gradient dL/dalpha:  {test_grad:.6e}")

In [ ]:
from scipy.optimize import minimize as scipy_minimize

# Start from a wrong initial guess: alpha=0.05 (true value is 0.02)
alpha_init = 0.05
history = {"alpha": [alpha_init], "loss": []}


def objective(alpha_arr):
    """Wrapper for scipy: returns (loss, grad) as plain floats."""
    loss, grad = grad_fn(float(alpha_arr[0]))
    history["alpha"].append(float(alpha_arr[0]))
    history["loss"].append(float(loss))
    return float(loss), np.array([float(grad)])


result = scipy_minimize(
    objective,
    x0=np.array([alpha_init]),
    method="L-BFGS-B",
    jac=True,
    bounds=[(1e-6, 0.1)],
    options={"maxiter": 100},
)

alpha_recovered = result.x[0]
print(f"Converged in {len(history['loss'])} iterations")
print(f"Recovered alpha: {alpha_recovered:.8f}")
print(f"True alpha:      {alpha_true}")
print(f"Final loss:      {result.fun:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curve
axes[0].semilogy(history["loss"])
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss (MSE)")
axes[0].set_title("Loss convergence")
axes[0].grid(True, alpha=0.3)

# Alpha convergence
axes[1].plot(history["alpha"], label="Estimated $\\alpha$")
axes[1].axhline(y=alpha_true, color="r", linestyle="--", label="True $\\alpha$")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("$\\alpha$")
axes[1].set_title("Parameter convergence")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Temperature profiles
T_recovered = simulate(T_init, alpha_recovered, n_steps)
T_initial_guess = simulate(T_init, alpha_init, n_steps)
axes[2].plot(x, T_observed, "k--", label="Observed", linewidth=2)
axes[2].plot(
    x, T_recovered, label=f"Recovered ($\\alpha$={alpha_recovered:.4f})", linewidth=2
)
axes[2].plot(
    x,
    T_initial_guess,
    ":",
    label=f"Initial guess ($\\alpha$={alpha_init})",
    linewidth=2,
    alpha=0.7,
)
axes[2].set_xlabel("x")
axes[2].set_ylabel("T")
axes[2].set_title("Temperature profiles")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

## Step 5: Verify gradient accuracy against finite differences

To confirm that Enzyme produces exact gradients, we compare against finite difference approximations at several step sizes. Enzyme's gradients should match to machine precision, while finite differences degrade for both too-large (truncation error) and too-small (roundoff error) step sizes.

In [ ]:
# Compare Enzyme gradient vs finite differences at a non-optimal alpha
# (At the optimum the gradient is zero, which makes relative error meaningless)
alpha_test = 0.03

_, enzyme_grad = grad_fn(alpha_test)

# Finite difference gradients at various step sizes
epsilons = np.logspace(-2, -12, 20)
fd_grads = []
for eps in epsilons:
    loss_plus = loss_fn(alpha_test + eps)
    loss_minus = loss_fn(alpha_test - eps)
    fd_grads.append(float((loss_plus - loss_minus) / (2 * eps)))

fd_grads = np.array(fd_grads)
rel_errors = np.abs(fd_grads - float(enzyme_grad)) / (
    np.abs(float(enzyme_grad)) + 1e-30
)

plt.figure(figsize=(8, 4))
plt.loglog(epsilons, rel_errors, "o-", label="FD vs Enzyme")
plt.xlabel("Finite difference step size $\\epsilon$")
plt.ylabel("Relative error vs. Enzyme gradient")
plt.title("Enzyme provides exact gradients; finite differences have a sweet spot")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

print(f"Enzyme gradient:         {float(enzyme_grad):.10e}")
print(f"Best FD gradient:        {fd_grads[np.argmin(rel_errors)]:.10e}")
print(f"Best FD relative error:  {rel_errors.min():.2e}")

## Takeaways

1. **Exact gradients from compiled Fortran, automatically.** Enzyme differentiated the Fortran heat solver at the LLVM IR level -- no adjoint code, no source modifications, no approximation.

2. **Full JAX composability.** By packaging the solver as a Tesseract and using tesseract-jax, we called `jax.grad` through hundreds of Fortran solver invocations. L-BFGS-B converged in just a handful of iterations -- the Fortran solver is just another differentiable function.

3. **Machine-precision accuracy.** The Enzyme gradients match finite differences at their best, and remain exact where finite differences break down.

4. **Language-agnostic.** Enzyme works on LLVM IR, so the same approach applies to C, C++, Rust -- any language with an LLVM frontend. The Fortran example here is just the beginning.

5. **Reproducible and portable.** The entire toolchain (LFortran, LLVM 19, Enzyme) is packaged in the Tesseract container. `tesseract build` produces the same differentiated binary on any machine.

In [ ]:
heat_tesseract.teardown()